# 🔧 ETL / Preprocessing Pipeline
**Proyecto Final | Data Science — Henry**

**Objetivo:** Ejecutar el pipeline de limpieza, transformación e ingeniería de features.
Genera los artefactos en `data/processed/` listos para el modelado.

**Outputs generados:**
- `data/processed/retail_clean.parquet` — Dataset limpio completo
- `data/processed/basket_matrix.parquet` — Matriz binaria para Apriori
- `data/processed/user_item_matrix.parquet` — Matriz para Collaborative Filtering
- `data/processed/product_features.parquet` — Features a nivel de producto

---

## 1. Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_raw_data, load_config
from src.data.preprocessor import run_cleaning_pipeline, get_aov_baseline
from src.features.builder import (
    build_basket_matrix,
    build_user_item_matrix,
    build_cooccurrence_matrix,
    build_product_features,
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
FIGURES_PATH = PROJECT_ROOT / 'outputs' / 'figures'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed'

cfg = load_config()
print('✅ Setup completado')

## 2. Carga de Datos Crudos

In [ ]:
df_raw = load_raw_data(config=cfg)
print(f'Shape raw: {df_raw.shape}')

## 3. Pipeline de Limpieza

Usamos `strategy='keep'` para customer_id en la primera pasada (para association rules).
Luego generamos una segunda versión con `strategy='drop'` para collaborative filtering.

In [ ]:
# --- Dataset para Association Rules (mantiene filas sin customer_id) ---
df_ar = run_cleaning_pipeline(
    df_raw.copy(),
    customer_strategy='keep',
    min_price=cfg['data']['min_price'],
    min_basket_items=cfg['data']['min_invoice_items'],
)
print(f'\nShape para Association Rules: {df_ar.shape}')

In [ ]:
# --- Dataset para Collaborative Filtering (requiere customer_id) ---
df_cf = run_cleaning_pipeline(
    df_raw.copy(),
    customer_strategy='drop',
    min_price=cfg['data']['min_price'],
    min_basket_items=cfg['data']['min_invoice_items'],
)
print(f'\nShape para Collaborative Filtering: {df_cf.shape}')

## 4. AOV Baseline (guardar como referencia)

In [ ]:
aov_value, order_values = get_aov_baseline(df_ar)

# Guardar métricas baseline
baseline_metrics = {
    'aov_baseline': round(aov_value, 2),
    'median_order_value': round(order_values.median(), 2),
    'total_orders': len(order_values),
    'total_revenue': round(order_values.sum(), 2),
}

import json
metrics_path = PROJECT_ROOT / 'outputs' / 'reports' / 'baseline_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(baseline_metrics, f, indent=2)

print('✅ Métricas baseline guardadas:', metrics_path)
print(json.dumps(baseline_metrics, indent=2))

## 5. Ingeniería de Características

In [ ]:
# --- Basket Matrix (para Apriori) ---
print('Construyendo basket matrix...')
basket_matrix = build_basket_matrix(df_ar)
print(f'Basket matrix shape: {basket_matrix.shape}')
basket_matrix.head(3)

In [ ]:
# --- User-Item Matrix (para Collaborative Filtering) ---
print('Construyendo user-item matrix...')
user_item_matrix, sparsity = build_user_item_matrix(df_cf, value_col='quantity')
print(f'User-item matrix shape: {user_item_matrix.shape}')
print(f'Sparsity: {sparsity:.2f}%')

In [ ]:
# --- Product Features ---
print('Generando product features...')
product_features = build_product_features(df_ar)
print(f'Product features shape: {product_features.shape}')
product_features.head(10)

## 6. Guardar Artefactos Procesados

In [ ]:
# Guardar en formato parquet (eficiente y preserva tipos)
df_ar.to_parquet(PROCESSED_PATH / 'retail_clean_ar.parquet', index=False)
df_cf.to_parquet(PROCESSED_PATH / 'retail_clean_cf.parquet', index=False)
basket_matrix.to_parquet(PROCESSED_PATH / 'basket_matrix.parquet')
user_item_matrix.to_parquet(PROCESSED_PATH / 'user_item_matrix.parquet')
product_features.to_parquet(PROCESSED_PATH / 'product_features.parquet')

print('✅ Todos los artefactos guardados en data/processed/')
print(f'  retail_clean_ar.parquet    : {df_ar.shape}')
print(f'  retail_clean_cf.parquet    : {df_cf.shape}')
print(f'  basket_matrix.parquet      : {basket_matrix.shape}')
print(f'  user_item_matrix.parquet   : {user_item_matrix.shape}')
print(f'  product_features.parquet   : {product_features.shape}')

## 7. Visualización Post-Limpieza
Comparación antes/después del pipeline de limpieza.

In [ ]:
FIGURES_PATH = PROJECT_ROOT / 'outputs' / 'figures'

# Comparativa de volumen de datos
stages = ['Raw', 'Clean (AR)', 'Clean (CF)']
counts = [len(df_raw), len(df_ar), len(df_cf)]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(stages, counts, color=['#d9534f', '#5bc0de', '#5cb85c'], edgecolor='white')
ax.set_title('Registros por Etapa del Pipeline de Limpieza')
ax.set_ylabel('Número de registros')
for bar, count in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(counts) * 0.01,
        f'{count:,}',
        ha='center', fontsize=11, fontweight='bold'
    )
plt.tight_layout()
fig.savefig(FIGURES_PATH / '10_data_volume_pipeline.png', dpi=150, bbox_inches='tight')
print(f'💾 Guardado: {FIGURES_PATH / "10_data_volume_pipeline.png"}')
plt.show()